In [1]:
!pip install transformers
!pip install torch
!pip install datasets
!pip install sentence-transformers


In [2]:
import pandas as pd
from datasets import load_dataset
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
from tqdm import tqdm


In [3]:
df_passage = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
df_test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

print(df_passage.head())
print(df_test.head())

                                                 passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
16083  We have studied the effects of curare on respo...
23188  Kinetic and electrophoretic properties of 230-...
23469  Male Wistar specific-pathogen-free rats aged 2...
                                             question  \
id                                                      
0   Is Hirschsprung disease a mendelian or a multi...   
1   List signaling molecules (ligands) that intera...   
2                    Is the protein Papilin secreted?   
3                   Are long non coding RNAs spliced?   
4                   Is RANKL secreted from the cells?   

                                               answer  \
id                                                      
0   Coding sequence mutations in RET, GDNF, EDNRB,...   
1   The 7 known EGFR ligands  

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
model_name = "dmis-lab/biobert-large-cased-v1.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Initialize HuggingFace's pipeline for text generation
chatbot = pipeline("text-generation", model=model, tokenizer=tokenizer)


config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


model.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Device set to use cpu


In [12]:
# Define a few-shot prompt
def build_few_shot_prompt(df, num_examples=3):
    prompt = "The following are questions and answers related to biomedical topics.\n\n"
    for i in range(num_examples):
        question = df.iloc[i]['question']
        answer = df.iloc[i]['answer']
        prompt += f"Q: {question}\nA: {answer}\n\n"
    prompt += "Q: {question}\nA:"
    return prompt


In [13]:
# Implement Chain of Thought (CoT)
def build_cot_prompt(question, context):
    prompt = f"Let's think step by step. The question is: {question}\n"
    prompt += f"Context: {context}\n"
    prompt += "Now, based on this context, let me answer:\n"
    prompt += f"Answer: "
    return prompt


In [19]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load sentence transformer model for embeddings
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Sample 10% of the passages from the DataFrame
df_sampled = df_passage.sample(frac=0.1, random_state=42)

# Index the sampled passages for fast retrieval
passage_embeddings = embedding_model.encode(df_sampled['passage'].tolist(), show_progress_bar=True)
index = faiss.IndexFlatL2(passage_embeddings.shape[1])
index.add(np.array(passage_embeddings))

# Function to get the most relevant context
def get_relevant_context(question, top_k=1):
    question_embedding = embedding_model.encode([question])
    distances, indices = index.search(question_embedding, top_k)
    context = df_passage['passage'].iloc[indices[0][0]]
    return context


Batches:   0%|          | 0/126 [00:00<?, ?it/s]

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Load model and tokenizer
model_name = "dmis-lab/biobert-large-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Initialize pipeline
chatbot = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Function to generate a response
def chatbot_response(question, max_length=200, max_new_tokens=256):
    # Tokenize input question and truncate if needed
    inputs = tokenizer(question, return_tensors="pt", truncation=True, max_length=max_length)

    # Generate a response
    response = chatbot(question, max_length=200, max_new_tokens=256, truncation=True)[0]['generated_text']

    return response.strip()

# Example usage
question = "What is the function of hemoglobin?"
response = chatbot_response(question)
print(response)


If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`
Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is the function of hemoglobin? plastered cellbrian Labs Streetbo ambush bred Pact Calling cello Crescent backyard bombings fence Katy attacksbage crawling ambushddling Bombbari cell Pact fencing coded neutrality painting Calling cradlebide rails cradleracy bombings cradleH1 bombings Bomb attacksnco bombingceded judged bombings Calling lowest Capitalrod judged armsbo earth dancedrma bombardmentwitz Guards bred Fiat communism attacks Street neighbor Job Bombput hunterrmarmabo attacks trap Attack Guns Gunsgado Germain cell counted Be lowest arms Earthbari offensive communism attacks ambush Metalrma attacksncobethoting rails lowestroft popzelrma NoiseH1imes communism posed fence modeledboept mathoidddling factionrdainiaoidoting daterma Keys Capital ho bombings lowest Mercury Battle Tech attackput dancedbo Bari Mouseoting Buchanan Gates Hatebo attacked Gates cell lowest Venus trap socialismrma Greeks attack Kamenining Castro everybody attackslava Outside Chao communism attack coded gan

In [8]:
# Simple function to check if the question is in the biomedical domain
def is_biomedical_question(question):
    biomedical_keywords = ['gene', 'protein', 'enzyme', 'disease', 'cancer', 'hemoglobin', 'cell', 'molecule']
    return any(keyword in question.lower() for keyword in biomedical_keywords)

# Modify chatbot response function
def chatbot_response_with_domain_check(question, use_few_shot=True, num_examples=3, use_cot=False):
    if not is_biomedical_question(question):
        return "Sorry, I can only answer biomedical questions."

    # Proceed with the regular chatbot response generation
    return chatbot_response(question, use_few_shot, num_examples, use_cot)

# Example usage
response = chatbot_response_with_domain_check("What is the capital of Canada?")
print(response)  # This will respond with "Sorry, I can only answer biomedical questions."


Sorry, I can only answer biomedical questions.


In [9]:
!pip install evaluate
!pip install bert-score


In [6]:
import evaluate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import numpy as np

# Load ROUGE and BERTScore metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# Load model and tokenizer with is_decoder=True
model_name = "dmis-lab/biobert-large-cased-v1.1"  # Use your chosen model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set is_decoder=True to use BertLMHeadModel as a standalone
model = AutoModelForCausalLM.from_pretrained(model_name, is_decoder=True)

# Initialize HuggingFace's text-generation pipeline
chatbot = pipeline("text-generation", model=model, tokenizer=tokenizer, device=-1)  # Use CPU

# Sample 10 examples from the test dataset for faster processing (you can change n as needed)
df_test_sampled = df_test.sample(n=10, random_state=42)  # Adjust n for faster evaluation

# Extract questions and answers from the sample
queries = df_test_sampled['question'].tolist()
references = df_test_sampled['answer'].tolist()

def generate_response(question, max_length=100):
    # Generate the response using the chatbot, specify max_length for truncation
    response = chatbot(question, max_new_tokens=50, max_length=max_length, truncation=True)[0]['generated_text']
    return response.strip()

def evaluate_model(queries, references, chatbot):
    predictions = []

    # Process queries one by one (sequentially)
    for question in queries:
        response = generate_response(question)
        predictions.append(response)

    # Compute ROUGE score
    rouge_score = rouge.compute(predictions=predictions, references=references)

    # Compute BERTScore
    bertscore_score = bertscore.compute(predictions=predictions, references=references, lang='en')

    return rouge_score, bertscore_score

# Run evaluation
rouge_score, bertscore_score = evaluate_model(queries, references, chatbot)

# Print results
print("ROUGE Score:", rouge_score)
print("BERTScore:", bertscore_score)


Device set to use cpu
Both `max_new_tokens` (=50) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes

ROUGE Score: {'rouge1': np.float64(0.1180606187565652), 'rouge2': np.float64(0.05254183911326768), 'rougeL': np.float64(0.10311212257005575), 'rougeLsum': np.float64(0.10389133190438796)}
BERTScore: {'precision': [0.7670921683311462, 0.7778249382972717, 0.7468589544296265, 0.7563706636428833, 0.769599974155426, 0.7709071040153503, 0.7553856372833252, 0.7788537740707397, 0.7906979322433472, 0.7795587778091431], 'recall': [0.7950648665428162, 0.8020673990249634, 0.8217618465423584, 0.7799246907234192, 0.9011441469192505, 0.7924092411994934, 0.8379063606262207, 0.8117238283157349, 0.8975714445114136, 0.8156933784484863], 'f1': [0.780828058719635, 0.789760172367096, 0.7825220823287964, 0.7679671049118042, 0.8301935195922852, 0.7815103530883789, 0.7945090532302856, 0.7949491739273071, 0.8407518863677979, 0.7972168326377869], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.53.2)'}
